# Stromal treeArches / scHPL Pipeline v1.1 (branch-wise)

**Purpose**: Run the **branch-wise** stromal scHPL/treeArches workflow on scArches-mapped query cells and compare transferred labels with branch-specific hierarchical validation  
**Mode**: branch-wise (`endothelial`, `fibroblast`, `smc`)  
**Input**: Reference h5ad (with scanvi latent + labels) + Query h5ad (output of v2.1)  
**Adds**: `schpl_branch`, `schpl_branch_source`, `schpl_pred_raw`, `schpl_pred`, `schpl_rejected`, `schpl_prob`, `schpl_reject_type` to query obs  
**Does NOT modify**: any existing obs columns from v2.1  
**Date**: 2026-04-06  
**Author**: r2end

---
## Unified methodology

- This notebook follows the shared `schpl` convention in this folder: scHPL is a **secondary hierarchical validator** after stromal scArches/scANVI mapping, not a replacement for the transferred labels.
- The feature space is the precomputed integrated latent (`X_scanvi` here); UMAP is visualization-only.
- Recommended scHPL setup kept here: **kNN** classifier, `dimred=True`, `useRE=True`, `n_neighbors=50`, `FN=0.5`, `rej_threshold=0.5`.
- `Rejected` means the cell is **not stably absorbed by the current branch hierarchy**; it is a follow-up flag, not an automatic novel-type conclusion.
- Cells outside the modeled branches are marked as `Ignored` rather than forced through scHPL.

## Design notes

- This notebook is the **branch-wise** variant of the folder-wide schpl methodology.
- Query branch assignment mirrors stromal scArches v2.1 workflow:
  - `fibroblast_cells.h5ad`  → fibroblast branch
  - `endothelial_cells.h5ad` → endothelial branch
  - `smc_cells.h5ad`         → smooth-muscle branch (**includes Pericyte**)
- Reference training is also split by lineage before scHPL fitting
- `Schwann` labels are **ignored** in branch-wise scHPL training
- scVI/scANVI params unchanged; scHPL operates on already-computed `X_scanvi`

## Rejection criteria (scHPL)

A modeled query cell is flagged as a follow-up / novel-state candidate if ANY of:
1. `Rejection (dist)` : too far from all reference neighbours in latent space
2. `Rejected (RE)`    : reconstruction error > threshold (PCA mismatch)
3. `Rejection (prob)` : max posterior probability < `rej_threshold` (0.5)

Cells outside the three modeled branches are marked as `Ignored` rather than forced through scHPL.

---

## Section 1: Imports

In [21]:
import os

for _k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_k] = "8"

import warnings
warnings.filterwarnings('ignore')

import gc
import time
import pickle
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import scanpy as sc

# scHPL modules
from scHPL import train as schpl_train
from scHPL import predict as schpl_predict
from scHPL import evaluate as schpl_evaluate
from scHPL import utils as schpl_utils

PIPELINE_START = time.time()
np.random.seed(42)

print("=" * 70)
print("Stromal treeArches / scHPL Pipeline v1.1 (branch-wise)")
print("=" * 70)
try:
    import scHPL
    print(f"scHPL version : {scHPL.__version__}")
except AttributeError:
    print("scHPL imported (version attr not available)")
print(f"scanpy        : {sc.__version__}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")

Stromal treeArches / scHPL Pipeline v1.1 (branch-wise)
scHPL imported (version attr not available)
scanpy        : 1.11.5
numpy         : 1.26.4
pandas        : 1.5.3


## Section 2: Configuration

**Edit zone**: update paths to match your actual files.

In [22]:
PIPELINE_VERSION = "v1.1-branchwise"
PIPELINE_TAG = "v1_1_branchwise"
PIPELINE_DATE = "2026-04-06"

# ===== INPUT PATHS =====

# Preferred branch-wise reference manifest generated by:
#   stromal_reintegration_branchwise_scvi_scanvi_20260407_v1_5.py
PATH_REF_MANIFEST = Path("/home/h2048/data/py/0407/stromal_reintegration_v1_5_branchwise/branch_reference_manifest.json")

# Legacy fallback: single reference h5ad (kept for backward compatibility)
PATH_REF_H5AD = Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3/stromal_reintegrated_scvi_scanvi_v1_3.h5ad")

# Preferred branch-wise query manifest generated by:
#   stromal_scarches_query_branchwise_20260407_v1_0.py
PATH_QRY_MANIFEST = Path("/home/h2048/data/py/0407/stromal_scarches_query_branchwise_v1_0/branch_query_manifest.json")

# Legacy fallback: global stromal query h5ad from v2.1
PATH_QRY_H5AD = Path("/home/h2048/data/py/0315/stromal_scarches_query_v2_1/adata_stromal_query_mapped_v2_1.h5ad")

# Output directory (new versioned path to avoid overwriting v1.0 outputs)
OUTPUT_DIR = Path(f"/home/h2048/data/py/0406/stromal_schpl_{PIPELINE_TAG}")
FIG_DIR = OUTPUT_DIR / "figures"
TREE_DIR = OUTPUT_DIR / "trees"
for _d in [OUTPUT_DIR, FIG_DIR, TREE_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ===== KEY NAMES — must match branch references =====

REF_LABEL_KEY = "scanvi_label"
REF_LATENT_KEY = "X_scanvi"
QRY_LATENT_KEY = "X_scanvi"
UNLABELED = "Unknown"

# L3 -> L2 mapping (copied from v2.1 exactly)
L3_TO_L2 = {
    "Endothelia_Lymphatic":                   "Endothelial",
    "Endothelia_vascular_Cap_a":              "Endothelial",
    "Endothelia_vascular_Cap_g":              "Endothelial",
    "Endothelia_vascular_arterial_pulmonary": "Endothelial",
    "Endothelia_vascular_arterial_systemic":  "Endothelial",
    "Endothelia_vascular_venous_pulmonary":   "Endothelial",
    "Endothelia_vascular_venous_systemic":    "Endothelial",
    "Fibro_adventitial":                      "Fibroblast",
    "Fibro_alveolar":                         "Fibroblast",
    "Fibro_myofibroblast":                    "Fibroblast",
    "Fibro_peribronchial":                    "Fibroblast",
    "Fibro_stress_activated":                 "Fibroblast",
    "Muscle_pericyte_pulmonary":              "Pericyte",
    "Muscle_pericyte_systemic":               "Pericyte",
    "Muscle_perivascular_immune_recruiting":  "Smooth_Muscle",
    "Muscle_smooth_arterial_systemic":        "Smooth_Muscle",
    "Muscle_smooth_pulmonary":                "Smooth_Muscle",
    "Schwann_nonmyelinating":                 "Schwann",
}

BRANCH_CONFIGS = {
    "endothelial": {
        "display": "Endothelial",
        "query_subset": "endothelial",
        "allowed_l2": ("Endothelial",),
    },
    "fibroblast": {
        "display": "Fibroblast",
        "query_subset": "fibroblast",
        "allowed_l2": ("Fibroblast",),
    },
    "smc": {
        "display": "Smooth_Muscle",
        "query_subset": "smc",
        "allowed_l2": ("Smooth_Muscle", "Pericyte"),
    },
}


def l3_to_branch(label):
    l2 = L3_TO_L2.get(str(label))
    if l2 == "Endothelial":
        return "endothelial"
    if l2 == "Fibroblast":
        return "fibroblast"
    if l2 in {"Smooth_Muscle", "Pericyte"}:
        return "smc"
    return None


def normalize_query_subset(value):
    value = str(value).strip().lower()
    mapping = {
        "endothelial": "endothelial",
        "fibroblast": "fibroblast",
        "smc": "smc",
        "smooth_muscle": "smc",
        "smooth muscle": "smc",
    }
    return mapping.get(value)


# ===== scHPL PARAMETERS =====
SCHPL_CLASSIFIER = "knn"
SCHPL_DIMRED = True
SCHPL_USE_RE = True
SCHPL_N_NEIGHBORS = 50
SCHPL_FN = 0.5
SCHPL_REJ_THRESHOLD = 0.5

# ===== OUTPUT OBS COLUMN NAMES =====
COL_SCHPL_BRANCH = "schpl_branch"
COL_SCHPL_BRANCH_SOURCE = "schpl_branch_source"
COL_SCHPL_RAW = "schpl_pred_raw"
COL_SCHPL_PRED = "schpl_pred"
COL_SCHPL_PROB = "schpl_prob"
COL_SCHPL_REJECTED = "schpl_rejected"
COL_SCHPL_REJ_TYPE = "schpl_reject_type"

# ===== VISUALIZATION =====
DPI = 300
FIG_FORMAT = "pdf"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.settings.vector_friendly = True

print("Configuration loaded")
print(f"  Pipeline version      : {PIPELINE_VERSION}")
print(f"  Reference manifest    : {PATH_REF_MANIFEST}")
print(f"  Legacy reference h5ad : {PATH_REF_H5AD}")
print(f"  Query manifest        : {PATH_QRY_MANIFEST}")
print(f"  Legacy query h5ad     : {PATH_QRY_H5AD}")
print(f"  Output dir            : {OUTPUT_DIR}")
print(f"  Tree dir              : {TREE_DIR}")
print(f"  Ref label key         : {REF_LABEL_KEY}")
print(f"  Ref latent key        : {REF_LATENT_KEY}")
print(f"  Qry latent key        : {QRY_LATENT_KEY}")
print(f"  scHPL classifier      : {SCHPL_CLASSIFIER}  dimred={SCHPL_DIMRED}  useRE={SCHPL_USE_RE}")
print("  Branch setup:")
for _branch_name, _cfg in BRANCH_CONFIGS.items():
    print(
        f"    {_branch_name:<12} -> display={_cfg['display']}, "
        f"query_subset={_cfg['query_subset']}, allowed_l2={list(_cfg['allowed_l2'])}"
    )

Configuration loaded
  Pipeline version      : v1.1-branchwise
  Reference manifest    : /home/h2048/data/py/0407/stromal_reintegration_v1_5_branchwise/branch_reference_manifest.json
  Legacy reference h5ad : /home/h2048/data/py/0308/stromal_reintegration_v1_3/stromal_reintegrated_scvi_scanvi_v1_3.h5ad
  Query manifest        : /home/h2048/data/py/0407/stromal_scarches_query_branchwise_v1_0/branch_query_manifest.json
  Legacy query h5ad     : /home/h2048/data/py/0315/stromal_scarches_query_v2_1/adata_stromal_query_mapped_v2_1.h5ad
  Output dir            : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise
  Tree dir              : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/trees
  Ref label key         : scanvi_label
  Ref latent key        : X_scanvi
  Qry latent key        : X_scanvi
  scHPL classifier      : knn  dimred=True  useRE=True
  Branch setup:
    endothelial  -> display=Endothelial, query_subset=endothelial, allowed_l2=['Endothelial']
    fibroblast   -> d

## Section 3: Load Reference Data

In [23]:
print("=" * 70)
print("STEP 1: LOAD REFERENCE")
print("=" * 70)

ref_manifest = None
ref_branch_data = {}
ref_total_cells = 0
ref_modeled_cells = 0

if PATH_REF_MANIFEST.exists():
    print(f"Loading branch reference manifest: {PATH_REF_MANIFEST}")
    with open(PATH_REF_MANIFEST) as f:
        ref_manifest = json.load(f)

    if "branches" not in ref_manifest:
        raise KeyError("[ERROR] Reference manifest missing 'branches' key")

    for branch_name, cfg in BRANCH_CONFIGS.items():
        if branch_name not in ref_manifest["branches"]:
            raise KeyError(f"[ERROR] Branch '{branch_name}' missing from manifest")

        branch_info = ref_manifest["branches"][branch_name]
        branch_h5ad = Path(branch_info["reference_h5ad"])
        if not branch_h5ad.exists():
            raise FileNotFoundError(f"[ERROR] Branch reference h5ad not found: {branch_h5ad}")

        ad_ref_branch = sc.read_h5ad(branch_h5ad)
        ref_total_cells += int(ad_ref_branch.n_obs)

        if REF_LATENT_KEY not in ad_ref_branch.obsm:
            raise KeyError(
                f"[ERROR] REF_LATENT_KEY='{REF_LATENT_KEY}' not in {branch_h5ad.name} obsm. "
                f"Available: {list(ad_ref_branch.obsm.keys())}"
            )
        if REF_LABEL_KEY not in ad_ref_branch.obs.columns:
            raise KeyError(
                f"[ERROR] REF_LABEL_KEY='{REF_LABEL_KEY}' not in {branch_h5ad.name} obs. "
                f"Available: {list(ad_ref_branch.obs.columns)}"
            )

        X_branch_all = ad_ref_branch.obsm[REF_LATENT_KEY].astype(np.float32)
        y_branch_all = ad_ref_branch.obs[REF_LABEL_KEY].astype(str)
        l2_branch_all = y_branch_all.map(L3_TO_L2).fillna("Unknown")
        keep_mask = (y_branch_all != UNLABELED) & l2_branch_all.isin(cfg["allowed_l2"])

        X_branch = X_branch_all[keep_mask.to_numpy()]
        y_branch = y_branch_all.loc[keep_mask].to_numpy()
        ref_modeled_cells += int(len(y_branch))

        if len(y_branch) == 0:
            raise ValueError(f"[ERROR] Branch '{branch_name}' has 0 usable labels after filtering")

        ref_branch_data[branch_name] = {
            "h5ad_path": branch_h5ad,
            "adata": ad_ref_branch,
            "X": X_branch,
            "y": y_branch,
            "n_total": int(ad_ref_branch.n_obs),
            "n_used": int(len(y_branch)),
            "n_labels": int(pd.Series(y_branch).nunique()),
        }

        print(
            f"  [{branch_name}] {branch_h5ad.name}: total={ad_ref_branch.n_obs:,}, "
            f"used={len(y_branch):,}, labels={pd.Series(y_branch).nunique()}"
        )
        for lbl, n in pd.Series(y_branch).value_counts().head(8).items():
            print(f"    {lbl}: {n:,}")

else:
    print(f"[WARN] Branch manifest not found; falling back to legacy single reference h5ad: {PATH_REF_H5AD}")
    if not PATH_REF_H5AD.exists():
        raise FileNotFoundError(f"[ERROR] Legacy reference h5ad not found: {PATH_REF_H5AD}")

    adata_ref = sc.read_h5ad(PATH_REF_H5AD)
    ref_total_cells = int(adata_ref.n_obs)
    print(f"  Legacy ref shape : {adata_ref.shape}")
    print(f"  Legacy ref obsm  : {list(adata_ref.obsm.keys())}")

    if REF_LATENT_KEY not in adata_ref.obsm:
        raise KeyError(
            f"[ERROR] REF_LATENT_KEY='{REF_LATENT_KEY}' not in legacy reference obsm. "
            f"Available: {list(adata_ref.obsm.keys())}"
        )
    if REF_LABEL_KEY not in adata_ref.obs.columns:
        raise KeyError(
            f"[ERROR] REF_LABEL_KEY='{REF_LABEL_KEY}' not in legacy reference obs. "
            f"Available: {list(adata_ref.obs.columns)}"
        )

    X_ref_all = adata_ref.obsm[REF_LATENT_KEY].astype(np.float32)
    y_ref_all = adata_ref.obs[REF_LABEL_KEY].astype(str)
    ref_branch_all = y_ref_all.map(l3_to_branch)

    for branch_name, cfg in BRANCH_CONFIGS.items():
        keep_mask = (y_ref_all != UNLABELED) & (ref_branch_all == branch_name)
        X_branch = X_ref_all[keep_mask.to_numpy()]
        y_branch = y_ref_all.loc[keep_mask].to_numpy()
        ref_modeled_cells += int(len(y_branch))

        if len(y_branch) == 0:
            raise ValueError(f"[ERROR] Legacy reference branch '{branch_name}' has 0 usable cells")

        ref_branch_data[branch_name] = {
            "h5ad_path": PATH_REF_H5AD,
            "adata": adata_ref,
            "X": X_branch,
            "y": y_branch,
            "n_total": int(keep_mask.sum()),
            "n_used": int(len(y_branch)),
            "n_labels": int(pd.Series(y_branch).nunique()),
        }

print(f"\nReference total cells   : {ref_total_cells:,}")
print(f"Reference modeled cells : {ref_modeled_cells:,}")
print("Branch-wise reference summary:")
for branch_name, info in ref_branch_data.items():
    print(
        f"  {branch_name:<12} : used={info['n_used']:,}, labels={info['n_labels']}, "
        f"source={Path(info['h5ad_path']).name}"
    )

STEP 1: LOAD REFERENCE
Loading branch reference manifest: /home/h2048/data/py/0407/stromal_reintegration_v1_5_branchwise/branch_reference_manifest.json
  [endothelial] adata_endothelial_reference_v1_5_branchwise.h5ad: total=34,362, used=26,957, labels=7
    Endothelia_vascular_venous_systemic: 15,930
    Endothelia_Lymphatic: 5,072
    Endothelia_vascular_Cap_g: 2,876
    Endothelia_vascular_arterial_pulmonary: 1,323
    Endothelia_vascular_venous_pulmonary: 753
    Endothelia_vascular_arterial_systemic: 745
    Endothelia_vascular_Cap_a: 258
  [fibroblast] adata_fibroblast_reference_v1_5_branchwise.h5ad: total=17,848, used=11,983, labels=5
    Fibro_adventitial: 6,506
    Fibro_peribronchial: 2,337
    Fibro_stress_activated: 1,630
    Fibro_alveolar: 1,357
    Fibro_myofibroblast: 153
  [smc] adata_smc_reference_v1_5_branchwise.h5ad: total=4,339, used=2,925, labels=5
    Muscle_pericyte_pulmonary: 1,105
    Muscle_smooth_pulmonary: 703
    Muscle_pericyte_systemic: 488
    Muscle_smo

## Section 4: Load Query Data

In [24]:
print("=" * 70)
print("STEP 2: LOAD QUERY")
print("=" * 70)

qry_manifest = None
qry_branch_data = {}
branch_query_indexers = {}
required_query_cols = [
    "cell_type_scarches_pred",
    "cell_type_scarches_final",
    "scarches_confidence",
    "scarches_margin",
]

if PATH_QRY_MANIFEST.exists():
    print(f"Loading branch query manifest: {PATH_QRY_MANIFEST}")
    with open(PATH_QRY_MANIFEST) as f:
        qry_manifest = json.load(f)

    if "branches" not in qry_manifest:
        raise KeyError("[ERROR] Query manifest missing 'branches' key")

    branch_adatas = {}
    for branch_name, cfg in BRANCH_CONFIGS.items():
        if branch_name not in qry_manifest["branches"]:
            raise KeyError(f"[ERROR] Branch '{branch_name}' missing from query manifest")

        branch_info = qry_manifest["branches"][branch_name]
        branch_h5ad = Path(branch_info["query_h5ad"])
        if not branch_h5ad.exists():
            raise FileNotFoundError(f"[ERROR] Branch query h5ad not found: {branch_h5ad}")

        ad_branch = sc.read_h5ad(branch_h5ad)
        if QRY_LATENT_KEY not in ad_branch.obsm:
            raise KeyError(
                f"[ERROR] QRY_LATENT_KEY='{QRY_LATENT_KEY}' not in {branch_h5ad.name} obsm. "
                f"Available: {list(ad_branch.obsm.keys())}"
            )
        missing_cols = [c for c in required_query_cols if c not in ad_branch.obs.columns]
        if missing_cols:
            raise KeyError(
                f"[ERROR] Branch query h5ad '{branch_h5ad.name}' missing required columns: {missing_cols}"
            )

        branch_adatas[branch_name] = ad_branch
        print(
            f"  [{branch_name}] {branch_h5ad.name}: shape={ad_branch.shape}, "
            f"accepted={(ad_branch.obs['cell_type_scarches_final'].astype(str) != UNLABELED).sum():,}"
        )

    if PATH_QRY_H5AD.exists():
        print("\n  [INFO] Using legacy global query h5ad as visualization scaffold.")
        print("         Branch-specific scArches/scANVI latents are loaded from branch_query_manifest.")
        adata_qry = sc.read_h5ad(PATH_QRY_H5AD)
    else:
        print("\n  [WARN] Legacy global query h5ad not found; concatenating branch query files as scaffold.")
        adata_qry = sc.concat(
            list(branch_adatas.values()),
            join="outer",
            merge="same",
            uns_merge="first",
            label="branch_query_source",
            keys=list(branch_adatas.keys()),
            index_unique=None,
            fill_value=0,
        )

    print(f"  Scaffold shape : {adata_qry.shape}")
    print(f"  Scaffold obsm  : {list(adata_qry.obsm.keys())}")

    for col in required_query_cols + ["query_subset"]:
        if col not in adata_qry.obs.columns:
            adata_qry.obs[col] = np.nan if col != "query_subset" else ""

    query_branch = pd.Series("unassigned", index=adata_qry.obs_names, dtype="object")
    branch_source = pd.Series("unassigned", index=adata_qry.obs_names, dtype="object")

    for branch_name, ad_branch in branch_adatas.items():
        branch_indexer = adata_qry.obs_names.get_indexer(ad_branch.obs_names)
        if np.any(branch_indexer < 0):
            missing_n = int((branch_indexer < 0).sum())
            raise KeyError(
                f"[ERROR] {missing_n} obs_names from branch '{branch_name}' are absent from the query scaffold"
            )

        branch_query_indexers[branch_name] = branch_indexer
        qry_branch_data[branch_name] = {
            "adata": ad_branch,
            "obs_names": ad_branch.obs_names.copy(),
            "indexer": branch_indexer,
        }

        query_branch.iloc[branch_indexer] = branch_name
        branch_source.iloc[branch_indexer] = "branch_query_manifest"

        for col in required_query_cols:
            adata_qry.obs.iloc[branch_indexer, adata_qry.obs.columns.get_loc(col)] = ad_branch.obs[col].astype(str).to_numpy() if ad_branch.obs[col].dtype == object else ad_branch.obs[col].to_numpy()

        if "query_subset" in ad_branch.obs.columns:
            adata_qry.obs.iloc[branch_indexer, adata_qry.obs.columns.get_loc("query_subset")] = ad_branch.obs["query_subset"].astype(str).to_numpy()
        else:
            adata_qry.obs.iloc[branch_indexer, adata_qry.obs.columns.get_loc("query_subset")] = branch_name

    adata_qry.obs[COL_SCHPL_BRANCH] = pd.Categorical(
        query_branch,
        categories=[*BRANCH_CONFIGS.keys(), "unassigned"],
    )
    adata_qry.obs[COL_SCHPL_BRANCH_SOURCE] = pd.Categorical(
        branch_source,
        categories=["branch_query_manifest", "query_subset", "predicted_l3", "final_label", "unassigned"],
    )

    print("\n  Query branch assignment source:")
    for src, n in adata_qry.obs[COL_SCHPL_BRANCH_SOURCE].value_counts(dropna=False).items():
        print(f"    {src}: {n:,}")

    print("\n  Branch assignment for branch-wise scHPL:")
    for branch_name in [*BRANCH_CONFIGS.keys(), "unassigned"]:
        n = int((adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == branch_name).sum())
        print(f"    {branch_name:<12} : {n:,}")

    print("\n  Branch-specific scArches final label distribution (top 8):")
    for lbl, n in adata_qry.obs["cell_type_scarches_final"].astype(str).value_counts().head(8).items():
        print(f"    {lbl}: {n:,}")

    n_unassigned = int((adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == "unassigned").sum())
    if n_unassigned > 0:
        print(
            f"\n  [WARN] {n_unassigned:,} query cells are not covered by branch_query_manifest and will be Ignored."
        )
    else:
        print("\n  [OK] All query cells loaded from branch_query_manifest")

else:
    print(f"[WARN] Query manifest not found; falling back to legacy single query h5ad: {PATH_QRY_H5AD}")
    if not PATH_QRY_H5AD.exists():
        raise FileNotFoundError(f"[ERROR] Query h5ad not found: {PATH_QRY_H5AD}")

    adata_qry = sc.read_h5ad(PATH_QRY_H5AD)
    print(f"  Shape  : {adata_qry.shape}")
    print(f"  obsm   : {list(adata_qry.obsm.keys())}")

    if QRY_LATENT_KEY not in adata_qry.obsm:
        raise KeyError(
            f"[ERROR] QRY_LATENT_KEY='{QRY_LATENT_KEY}' not in query obsm.\n"
            f"  Available: {list(adata_qry.obsm.keys())}"
        )

    missing_cols = [c for c in required_query_cols if c not in adata_qry.obs.columns]
    if missing_cols:
        raise ValueError(
            f"[ERROR] Query is missing v2.1 output columns: {missing_cols}\n"
            f"  Ensure PATH_QRY_H5AD points to adata_stromal_query_mapped_v2_1.h5ad"
        )

    new_cols = [
        COL_SCHPL_BRANCH,
        COL_SCHPL_BRANCH_SOURCE,
        COL_SCHPL_RAW,
        COL_SCHPL_PRED,
        COL_SCHPL_PROB,
        COL_SCHPL_REJECTED,
        COL_SCHPL_REJ_TYPE,
    ]
    collisions = [c for c in new_cols if c in adata_qry.obs.columns]
    if collisions:
        print(f"  [INFO] Overwriting existing schpl columns: {collisions}")

    branch_from_subset = pd.Series(index=adata_qry.obs_names, dtype="object")
    branch_source = pd.Series("unassigned", index=adata_qry.obs_names, dtype="object")

    if "query_subset" in adata_qry.obs.columns:
        branch_from_subset = adata_qry.obs["query_subset"].astype(str).map(normalize_query_subset)
    else:
        branch_from_subset[:] = None

    branch_from_pred = adata_qry.obs["cell_type_scarches_pred"].astype(str).map(l3_to_branch)
    branch_from_final = adata_qry.obs["cell_type_scarches_final"].astype(str).map(l3_to_branch)

    query_branch = branch_from_subset.copy()
    mask_subset = query_branch.notna()
    branch_source.loc[mask_subset] = "query_subset"

    fallback_pred = query_branch.isna() & branch_from_pred.notna()
    query_branch.loc[fallback_pred] = branch_from_pred.loc[fallback_pred]
    branch_source.loc[fallback_pred] = "predicted_l3"

    fallback_final = query_branch.isna() & branch_from_final.notna()
    query_branch.loc[fallback_final] = branch_from_final.loc[fallback_final]
    branch_source.loc[fallback_final] = "final_label"

    query_branch = query_branch.fillna("unassigned")
    adata_qry.obs[COL_SCHPL_BRANCH] = pd.Categorical(
        query_branch,
        categories=[*BRANCH_CONFIGS.keys(), "unassigned"],
    )
    adata_qry.obs[COL_SCHPL_BRANCH_SOURCE] = pd.Categorical(
        branch_source,
        categories=["branch_query_manifest", "query_subset", "predicted_l3", "final_label", "unassigned"],
    )

    print(f"\n  Query latent shape        : {adata_qry.obsm[QRY_LATENT_KEY].shape}")
    print("  scArches final label distribution (top 8):")
    for lbl, n in adata_qry.obs["cell_type_scarches_final"].value_counts().head(8).items():
        print(f"    {lbl}: {n:,}")

    print("\n  Branch assignment for branch-wise scHPL:")
    for branch_name in [*BRANCH_CONFIGS.keys(), "unassigned"]:
        n = int((adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == branch_name).sum())
        print(f"    {branch_name:<12} : {n:,}")

    print("\n  Branch source:")
    for src, n in adata_qry.obs[COL_SCHPL_BRANCH_SOURCE].value_counts(dropna=False).items():
        print(f"    {src}: {n:,}")

    n_unassigned = int((adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == "unassigned").sum())
    if n_unassigned > 0:
        print(
            f"\n  [WARN] {n_unassigned:,} query cells could not be assigned to endothelial/fibroblast/smc. "
            "They will be marked as Ignored in downstream scHPL."
        )
    else:
        print("\n  [OK] All query cells assigned to a modeled branch")

STEP 2: LOAD QUERY
Loading branch query manifest: /home/h2048/data/py/0407/stromal_scarches_query_branchwise_v1_0/branch_query_manifest.json


  [endothelial] adata_endothelial_query_v1_0_branchwise_query.h5ad: shape=(29960, 4004), accepted=28,798
  [fibroblast] adata_fibroblast_query_v1_0_branchwise_query.h5ad: shape=(32912, 4000), accepted=32,386
  [smc] adata_smc_query_v1_0_branchwise_query.h5ad: shape=(19424, 4000), accepted=18,814

  [INFO] Using legacy global query h5ad as visualization scaffold.
         Branch-specific scArches/scANVI latents are loaded from branch_query_manifest.
  Scaffold shape : (82296, 4000)
  Scaffold obsm  : ['X_scanvi', 'X_scvi', 'X_umap']

  Query branch assignment source:
    branch_query_manifest: 82,296
    query_subset: 0
    predicted_l3: 0
    final_label: 0
    unassigned: 0

  Branch assignment for branch-wise scHPL:
    endothelial  : 29,960
    fibroblast   : 32,912
    smc          : 19,424
    unassigned   : 0

  Branch-specific scArches final label distribution (top 8):
    Fibro_peribronchial: 19,084
    Muscle_pericyte_systemic: 11,642
    Endothelia_vascular_Cap_g: 10,155
    

## Section 5: Build Initial Tree + Train scHPL Classifier

In [25]:
print("=" * 70)
print("STEP 3: TRAIN BRANCH-WISE scHPL TREES (reference only)")
print("=" * 70)

trained_trees = {}
trained_tree_paths = {}
branch_training_summary = []

for branch_name, cfg in BRANCH_CONFIGS.items():
    print("\n" + "-" * 70)
    print(f"BRANCH: {branch_name}  ({cfg['display']})")
    print("-" * 70)

    if branch_name not in ref_branch_data:
        print("  [WARN] Missing reference data for this branch; skipping")
        continue

    X_ref_branch = ref_branch_data[branch_name]["X"]
    y_ref_branch = ref_branch_data[branch_name]["y"]

    if X_ref_branch.shape[0] == 0:
        print("  [WARN] No reference cells in this branch; skipping")
        continue

    unique_labels = np.sort(np.unique(y_ref_branch))
    class_counts = pd.Series(y_ref_branch).value_counts()
    small_classes = class_counts[class_counts < SCHPL_N_NEIGHBORS]

    print(f"  Reference source  : {ref_branch_data[branch_name]['h5ad_path']}")
    print(f"  Reference cells   : {X_ref_branch.shape[0]:,}")
    print(f"  Unique labels     : {len(unique_labels)}")
    print(f"  Latent shape      : {X_ref_branch.shape}")
    print("  Labels (top 10):")
    for lbl, n in class_counts.head(10).items():
        print(f"    {lbl}: {n:,}")

    if len(small_classes) > 0:
        print(f"  [INFO] {len(small_classes)} classes have <{SCHPL_N_NEIGHBORS} cells:")
        for lbl, n in small_classes.items():
            print(f"    {lbl}: {n}")
        print("  dynamic_neighbors=True will handle these automatically.")

    tree_newick = f"({','.join(unique_labels)})root;"
    tree_init = schpl_utils.create_tree(tree_newick)

    t0 = time.time()
    tree_trained = schpl_train.train_tree(
        data=X_ref_branch,
        labels=y_ref_branch,
        tree=tree_init,
        classifier=SCHPL_CLASSIFIER,
        dimred=SCHPL_DIMRED,
        useRE=SCHPL_USE_RE,
        FN=SCHPL_FN,
        n_neighbors=SCHPL_N_NEIGHBORS,
        dynamic_neighbors=True,
    )
    elapsed_train = time.time() - t0

    tree_path = TREE_DIR / f"schpl_tree_{branch_name}_{PIPELINE_TAG}.pkl"
    with open(tree_path, "wb") as f:
        pickle.dump(tree_trained, f)

    trained_trees[branch_name] = tree_trained
    trained_tree_paths[branch_name] = tree_path
    branch_training_summary.append({
        "branch": branch_name,
        "display": cfg["display"],
        "reference_h5ad": str(ref_branch_data[branch_name]["h5ad_path"]),
        "n_cells": int(X_ref_branch.shape[0]),
        "n_labels": int(len(unique_labels)),
        "n_small_classes": int(len(small_classes)),
        "elapsed_s": round(elapsed_train, 2),
        "tree_path": str(tree_path),
    })

    print(f"  [OK] Training done : {elapsed_train:.1f} s")
    print(f"  [OK] Tree saved    : {tree_path}")

if len(trained_trees) == 0:
    raise RuntimeError("[ERROR] No branch-specific scHPL tree was trained successfully.")

branch_training_df = pd.DataFrame(branch_training_summary).sort_values("branch")
print("\nBranch training summary:")
print(branch_training_df.to_string(index=False))

STEP 3: TRAIN BRANCH-WISE scHPL TREES (reference only)

----------------------------------------------------------------------
BRANCH: endothelial  (Endothelial)
----------------------------------------------------------------------
  Reference source  : /home/h2048/data/py/0407/stromal_reintegration_v1_5_branchwise/endothelial/adata_endothelial_reference_v1_5_branchwise.h5ad
  Reference cells   : 26,957
  Unique labels     : 7
  Latent shape      : (26957, 75)
  Labels (top 10):
    Endothelia_vascular_venous_systemic: 15,930
    Endothelia_Lymphatic: 5,072
    Endothelia_vascular_Cap_g: 2,876
    Endothelia_vascular_arterial_pulmonary: 1,323
    Endothelia_vascular_venous_pulmonary: 753
    Endothelia_vascular_arterial_systemic: 745
    Endothelia_vascular_Cap_a: 258
  [OK] Training done : 22.9 s
  [OK] Tree saved    : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/trees/schpl_tree_endothelial_v1_1_branchwise.pkl

--------------------------------------------------------------

## Section 6: Predict Labels on Query

In [26]:
print("=" * 70)
print("STEP 4: PREDICT LABELS ON QUERY (branch-wise)")
print("=" * 70)

n_qry = adata_qry.n_obs
y_pred_raw_all = np.full(n_qry, "Unassigned", dtype=object)
y_prob_all = np.full(n_qry, np.nan, dtype=np.float32)
branch_prediction_summary = []

for branch_name, cfg in BRANCH_CONFIGS.items():
    if branch_name not in trained_trees:
        print(f"\n  [WARN] No trained tree found for branch '{branch_name}'; leaving cells as Unassigned")
        branch_prediction_summary.append({
            "branch": branch_name,
            "display": cfg["display"],
            "n_query_cells": 0,
            "elapsed_s": 0.0,
            "status": "missing_tree",
            "query_source": "none",
        })
        continue

    if branch_name in qry_branch_data:
        branch_adata = qry_branch_data[branch_name]["adata"]
        branch_indexer = np.asarray(qry_branch_data[branch_name]["indexer"], dtype=int)
        X_qry_branch = branch_adata.obsm[QRY_LATENT_KEY].astype(np.float32)
        query_source = "branch_query_manifest"
        n_branch = int(len(branch_indexer))
    else:
        mask_branch = adata_qry.obs[COL_SCHPL_BRANCH].astype(str).to_numpy() == branch_name
        branch_indexer = np.where(mask_branch)[0]
        n_branch = int(len(branch_indexer))
        if n_branch > 0:
            X_qry_branch = adata_qry.obsm[QRY_LATENT_KEY][branch_indexer].astype(np.float32)
        else:
            X_qry_branch = None
        query_source = "legacy_query_h5ad"

    if n_branch == 0:
        print(f"\n  [INFO] Branch '{branch_name}' has 0 query cells; skipping prediction")
        branch_prediction_summary.append({
            "branch": branch_name,
            "display": cfg["display"],
            "n_query_cells": 0,
            "elapsed_s": 0.0,
            "status": "no_query_cells",
            "query_source": query_source,
        })
        continue

    print(f"\n  Predicting branch '{branch_name}' on {n_branch:,} query cells...")
    print(f"  Query latent source: {query_source}")
    t0 = time.time()
    y_pred_branch, y_prob_branch = schpl_predict.predict_labels(
        X_qry_branch,
        tree=trained_trees[branch_name],
        threshold=SCHPL_REJ_THRESHOLD,
    )
    elapsed_pred = time.time() - t0

    y_pred_branch = np.asarray(y_pred_branch, dtype=str)
    y_pred_raw_all[branch_indexer] = y_pred_branch

    if y_prob_branch is not None:
        y_prob_branch = np.asarray(y_prob_branch, dtype=np.float32)
        if y_prob_branch.ndim == 2:
            y_prob_branch = y_prob_branch[:, 0]
        y_prob_all[branch_indexer] = y_prob_branch

    pred_counts = pd.Series(y_pred_branch).value_counts()
    print(f"  [OK] Prediction done: {elapsed_pred:.1f} s")
    print("  Raw prediction distribution:")
    for lbl, n in pred_counts.items():
        flag = "  <- REJECTED" if "Reject" in str(lbl) or str(lbl) in {"root", "root2"} else ""
        print(f"    {lbl}: {n:,}{flag}")

    branch_prediction_summary.append({
        "branch": branch_name,
        "display": cfg["display"],
        "n_query_cells": n_branch,
        "elapsed_s": round(elapsed_pred, 2),
        "status": "ok",
        "query_source": query_source,
    })

y_pred_raw = y_pred_raw_all
y_prob_raw = y_prob_all

n_unassigned = int((adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == "unassigned").sum())
if n_unassigned > 0:
    print(
        f"\n  [INFO] {n_unassigned:,} cells remain outside endothelial/fibroblast/smc branches "
        "and stay as 'Unassigned'."
    )

branch_prediction_df = pd.DataFrame(branch_prediction_summary).sort_values("branch")

STEP 4: PREDICT LABELS ON QUERY (branch-wise)

  Predicting branch 'endothelial' on 29,960 query cells...
  Query latent source: branch_query_manifest


  [OK] Prediction done: 126.1 s
  Raw prediction distribution:
    Endothelia_vascular_venous_systemic: 15,283
    Rejection (dist): 11,366  <- REJECTED
    Rejected (RE): 907  <- REJECTED
    Endothelia_vascular_arterial_systemic: 602
    Endothelia_vascular_Cap_g: 589
    root: 476  <- REJECTED
    Endothelia_Lymphatic: 392
    Endothelia_vascular_arterial_pulmonary: 275
    Endothelia_vascular_venous_pulmonary: 68
    Endothelia_vascular_Cap_a: 2

  Predicting branch 'fibroblast' on 32,912 query cells...
  Query latent source: branch_query_manifest
  [OK] Prediction done: 171.9 s
  Raw prediction distribution:
    Fibro_peribronchial: 17,429
    Fibro_adventitial: 6,548
    Fibro_alveolar: 3,179
    root: 3,001  <- REJECTED
    Rejection (dist): 1,932  <- REJECTED
    Fibro_myofibroblast: 422
    Rejected (RE): 380  <- REJECTED
    Fibro_stress_activated: 21

  Predicting branch 'smc' on 19,424 query cells...
  Query latent source: branch_query_manifest
  [OK] Prediction done: 67.4 

## Section 7: Process Rejection Types + Write to Query obs

In [27]:
print("=" * 70)
print("STEP 5: PROCESS BRANCH-WISE scHPL OUTPUT + WRITE TO OBS")
print("=" * 70)

y_pred_arr = np.asarray(y_pred_raw, dtype=str)
mask_ignored = y_pred_arr == "Unassigned"
mask_rej_dist = np.array(["dist" in s for s in y_pred_arr])
mask_rej_re = np.array(["RE)" in s for s in y_pred_arr])
mask_rej_prob = np.array([
    (not ignored) and (
        ("prob" in s) or
        (s in {"root", "root2"}) or
        ("Reject" in s and "dist" not in s and "RE" not in s)
    )
    for s, ignored in zip(y_pred_arr, mask_ignored)
])
mask_rejected = mask_rej_dist | mask_rej_re | mask_rej_prob
mask_accepted = (~mask_rejected) & (~mask_ignored)

data_qry_branch = adata_qry.obs[COL_SCHPL_BRANCH].astype(str)
adata_qry.obs[COL_SCHPL_RAW] = y_pred_arr

y_pred_clean = y_pred_arr.copy()
y_pred_clean[mask_rejected] = "Rejected"
y_pred_clean[mask_ignored] = "Ignored"
adata_qry.obs[COL_SCHPL_PRED] = y_pred_clean

y_prob_arr = np.asarray(y_prob_raw, dtype=np.float32)
if y_prob_arr.ndim == 2:
    y_prob_arr = y_prob_arr[:, 0]
adata_qry.obs[COL_SCHPL_PROB] = y_prob_arr
adata_qry.obs[COL_SCHPL_REJECTED] = mask_rejected

rej_type = np.where(
    mask_ignored,
    "branch_skip",
    np.where(
        mask_rej_dist,
        "dist",
        np.where(mask_rej_re, "RE", np.where(mask_rej_prob, "prob", "accepted"))
    ),
)
adata_qry.obs[COL_SCHPL_REJ_TYPE] = rej_type

n_total = adata_qry.n_obs
n_ignored = int(mask_ignored.sum())
n_rejected = int(mask_rejected.sum())
n_accepted = int(mask_accepted.sum())
n_modeled = n_total - n_ignored
rej_rate = (n_rejected / n_modeled * 100) if n_modeled > 0 else np.nan

print(f"\n  Total query cells      : {n_total:,}")
print(f"  Modeled by scHPL       : {n_modeled:,}")
print(f"  Ignored (branch skip)  : {n_ignored:,}")
print(f"  Accepted               : {n_accepted:,} ({(n_accepted / n_modeled * 100) if n_modeled else 0:.1f}%)")
print(f"  Rejected (all)         : {n_rejected:,} ({rej_rate if n_modeled else 0:.1f}%)")
print(f"    dist rejection       : {int(mask_rej_dist.sum()):,}")
print(f"    RE rejection         : {int(mask_rej_re.sum()):,}")
print(f"    prob rejection       : {int(mask_rej_prob.sum()):,}")

if n_modeled > 0:
    if rej_rate > 30:
        print("\n  [WARN] >30% rejection rate among modeled cells. Possible reasons:")
        print("    - Query contains genuine novel cell types")
        print("    - Branch assignment / reference lineage mismatch")
        print("    - Poor batch alignment between ref and query latent spaces")
    elif rej_rate < 1:
        print("\n  [INFO] <1% rejection among modeled cells. Query closely matches reference.")
    else:
        print(f"\n  [OK] Rejection rate {rej_rate:.1f}% among modeled cells")

print("\n  Final schpl_pred distribution:")
for lbl, n in adata_qry.obs[COL_SCHPL_PRED].value_counts().items():
    print(f"    {lbl}: {n:,}")

branch_rejection_summary = (
    adata_qry.obs.assign(
        _accepted=mask_accepted,
        _rejected=mask_rejected,
        _ignored=mask_ignored,
    )
    .groupby(COL_SCHPL_BRANCH, observed=False)
    .agg(
        n_total=(COL_SCHPL_PRED, "size"),
        n_accepted=("_accepted", "sum"),
        n_rejected=("_rejected", "sum"),
        n_ignored=("_ignored", "sum"),
    )
)
branch_rejection_summary["n_modeled"] = (
    branch_rejection_summary["n_total"] - branch_rejection_summary["n_ignored"]
)
branch_rejection_summary["rej_rate_pct"] = np.where(
    branch_rejection_summary["n_modeled"] > 0,
    branch_rejection_summary["n_rejected"] / branch_rejection_summary["n_modeled"] * 100,
    np.nan,
)

print("\n  Branch-wise rejection summary:")
print(branch_rejection_summary.to_string())

STEP 5: PROCESS BRANCH-WISE scHPL OUTPUT + WRITE TO OBS

  Total query cells      : 82,296
  Modeled by scHPL       : 82,296
  Ignored (branch skip)  : 0
  Accepted               : 62,690 (76.2%)
  Rejected (all)         : 19,606 (23.8%)
    dist rejection       : 13,360
    RE rejection         : 1,361
    prob rejection       : 4,885

  [OK] Rejection rate 23.8% among modeled cells

  Final schpl_pred distribution:
    Rejected: 19,606
    Fibro_peribronchial: 17,429
    Endothelia_vascular_venous_systemic: 15,283
    Muscle_pericyte_systemic: 9,956
    Fibro_adventitial: 6,548
    Muscle_smooth_arterial_systemic: 3,972
    Fibro_alveolar: 3,179
    Muscle_smooth_pulmonary: 1,895
    Muscle_pericyte_pulmonary: 1,055
    Muscle_perivascular_immune_recruiting: 1,002
    Endothelia_vascular_arterial_systemic: 602
    Endothelia_vascular_Cap_g: 589
    Fibro_myofibroblast: 422
    Endothelia_Lymphatic: 392
    Endothelia_vascular_arterial_pulmonary: 275
    Endothelia_vascular_venous_pul

## Section 8: Compare scHPL vs scANVI Predictions

In [28]:
print("=" * 70)
print("STEP 6: COMPARE scHPL vs scANVI PREDICTIONS")
print("=" * 70)

df_compare = adata_qry.obs[[
    COL_SCHPL_BRANCH,
    COL_SCHPL_BRANCH_SOURCE,
    "cell_type_scarches_final",
    "cell_type_scarches_pred",
    "scarches_confidence",
    COL_SCHPL_PRED,
    COL_SCHPL_REJECTED,
    COL_SCHPL_PROB,
    COL_SCHPL_REJ_TYPE,
]].copy()

df_modeled = df_compare[df_compare[COL_SCHPL_PRED] != "Ignored"].copy()
df_accepted = df_modeled[~df_modeled[COL_SCHPL_REJECTED]].copy()

if len(df_accepted) == 0:
    agree_rate = np.nan
    agreed = pd.Series(dtype=bool)
    print("  [WARN] No accepted modeled cells available for agreement analysis")
else:
    agreed = (
        df_accepted["cell_type_scarches_pred"].astype(str) ==
        df_accepted[COL_SCHPL_PRED].astype(str)
    )
    agree_rate = agreed.mean() * 100
    print(f"  Modeled cells (all)          : {len(df_modeled):,}")
    print(f"  Accepted modeled cells       : {len(df_accepted):,}")
    print(f"  Agreement (scANVI vs scHPL)  : {agree_rate:.1f}%")
    print(f"  Disagreement count           : {(~agreed).sum():,}")

branch_agreement_rows = []
for branch_name, cfg in BRANCH_CONFIGS.items():
    sub = df_accepted[df_accepted[COL_SCHPL_BRANCH].astype(str) == branch_name]
    if len(sub) == 0:
        branch_agreement_rows.append({
            "branch": branch_name,
            "display": cfg["display"],
            "n_accepted": 0,
            "agreement_pct": np.nan,
        })
        continue

    sub_agreed = (
        sub["cell_type_scarches_pred"].astype(str) ==
        sub[COL_SCHPL_PRED].astype(str)
    )
    rate = sub_agreed.mean() * 100
    branch_agreement_rows.append({
        "branch": branch_name,
        "display": cfg["display"],
        "n_accepted": int(len(sub)),
        "agreement_pct": round(rate, 2),
    })
    print(f"    {branch_name:<12} : accepted={len(sub):,}  agreement={rate:.1f}%")

branch_agreement_df = pd.DataFrame(branch_agreement_rows).sort_values("branch")

df_disagree = df_accepted.loc[~agreed].copy() if len(df_accepted) > 0 else pd.DataFrame()
pairs = pd.DataFrame(columns=[COL_SCHPL_BRANCH, "cell_type_scarches_pred", COL_SCHPL_PRED, "n"])
if len(df_disagree) > 0:
    print("\n  Top disagreement pairs (branch | scANVI_pred -> scHPL_pred):")
    pairs = (
        df_disagree.groupby([
            COL_SCHPL_BRANCH,
            "cell_type_scarches_pred",
            COL_SCHPL_PRED,
        ])
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
    )
    for _, row in pairs.head(12).iterrows():
        print(
            f"    [{row[COL_SCHPL_BRANCH]}] {row['cell_type_scarches_pred']} "
            f"-> {row[COL_SCHPL_PRED]}: {row['n']:,}"
        )

df_rejected = df_modeled[df_modeled[COL_SCHPL_REJECTED]].copy()
if len(df_rejected) > 0:
    print("\n  scANVI label distribution for scHPL-rejected cells:")
    for lbl, n in df_rejected["cell_type_scarches_final"].value_counts().head(10).items():
        print(f"    {lbl}: {n:,}")
    print("  [Note] Branch-specific rejections concentrated in one scANVI label can indicate")
    print("         lineage-consistent novel states rather than global reference mismatch.")

STEP 6: COMPARE scHPL vs scANVI PREDICTIONS
  Modeled cells (all)          : 82,296
  Accepted modeled cells       : 62,690
  Agreement (scANVI vs scHPL)  : 69.2%
  Disagreement count           : 19,327
    endothelial  : accepted=17,211  agreement=40.3%
    fibroblast   : accepted=27,599  agreement=76.7%
    smc          : accepted=17,880  agreement=85.4%

  Top disagreement pairs (branch | scANVI_pred -> scHPL_pred):
    [endothelial] Endothelia_vascular_Cap_g -> Endothelia_vascular_venous_systemic: 4,953
    [endothelial] Endothelia_vascular_Cap_a -> Endothelia_vascular_venous_systemic: 1,769
    [endothelial] Endothelia_vascular_venous_pulmonary -> Endothelia_vascular_venous_systemic: 1,561
    [fibroblast] Fibro_peribronchial -> Fibro_adventitial: 1,366
    [fibroblast] Fibro_alveolar -> Fibro_peribronchial: 1,120
    [endothelial] Endothelia_vascular_arterial_systemic -> Endothelia_vascular_venous_systemic: 1,006
    [fibroblast] Fibro_alveolar -> Fibro_adventitial: 988
    [fibr

## Section 9: Novel Cell Type Candidate Analysis

In [29]:
print("=" * 70)
print("STEP 7: NOVEL CELL TYPE CANDIDATE ANALYSIS")
print("=" * 70)

# Identify Leiden clusters enriched in rejected cells
# Use existing UMAP from v2.1 (X_umap in obsm)

if "X_umap" not in adata_qry.obsm:
    print("  [INFO] No X_umap in query; computing from X_scanvi...")
    sc.pp.neighbors(adata_qry, use_rep=QRY_LATENT_KEY, n_neighbors=30,
                    random_state=42, key_added="neighbors_scanvi")
    sc.tl.umap(adata_qry, neighbors_key="neighbors_scanvi",
               random_state=42, min_dist=0.5)
    print("  [OK] UMAP computed")
else:
    print("  [OK] Using existing X_umap from v2.1")

if "neighbors_scanvi" not in adata_qry.uns:
    sc.pp.neighbors(adata_qry, use_rep=QRY_LATENT_KEY, n_neighbors=30,
                    random_state=42, key_added="neighbors_scanvi")

sc.tl.leiden(adata_qry, resolution=0.5, key_added="leiden_schpl_qc",
             neighbors_key="neighbors_scanvi", random_state=42)
print(f"  Leiden clusters (res=0.5): {adata_qry.obs['leiden_schpl_qc'].nunique()}")

cluster_rej = (
    adata_qry.obs.groupby("leiden_schpl_qc")[COL_SCHPL_REJECTED]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "n_rejected", "count": "n_total", "mean": "rej_rate"})
    .sort_values("rej_rate", ascending=False)
)
cluster_rej["rej_rate_pct"] = (cluster_rej["rej_rate"] * 100).round(1)

NOVEL_REJ_RATE_THRESHOLD = 0.50
NOVEL_MIN_CELLS = 50

novel_clusters = cluster_rej[
    (cluster_rej["rej_rate"] > NOVEL_REJ_RATE_THRESHOLD) &
    (cluster_rej["n_rejected"] >= NOVEL_MIN_CELLS)
].index.tolist()

adata_qry.obs["schpl_novel_candidate"] = (
    adata_qry.obs["leiden_schpl_qc"].isin(novel_clusters) &
    adata_qry.obs[COL_SCHPL_REJECTED]
)

print(f"\n  Novel candidate clusters (rej_rate>{NOVEL_REJ_RATE_THRESHOLD*100:.0f}%, n>={NOVEL_MIN_CELLS}):")
if len(novel_clusters) == 0:
    print("    None detected. Rejected cells are scattered across clusters.")
    print("    -> Likely technical noise rather than novel populations.")
else:
    for c in novel_clusters:
        row = cluster_rej.loc[c]
        print(f"    Cluster {c}: {int(row['n_total']):,} cells, "
              f"{int(row['n_rejected']):,} rejected ({row['rej_rate_pct']:.1f}%)")
    n_novel_cells = adata_qry.obs["schpl_novel_candidate"].sum()
    print(f"  Total novel candidate cells: {n_novel_cells:,}")

print("\n  Cluster rejection rate summary:")
print(cluster_rej.head(10).to_string())

cluster_rej.to_csv(OUTPUT_DIR / f"cluster_rejection_summary_{PIPELINE_TAG}.csv")
print(f"\n  [OK] cluster_rejection_summary_{PIPELINE_TAG}.csv saved")

STEP 7: NOVEL CELL TYPE CANDIDATE ANALYSIS
  [OK] Using existing X_umap from v2.1
  Leiden clusters (res=0.5): 11

  Novel candidate clusters (rej_rate>50%, n>=50):
    Cluster 3: 10,373 cells, 7,242 rejected (69.8%)
  Total novel candidate cells: 7,242

  Cluster rejection rate summary:
                 n_rejected  n_total  rej_rate  rej_rate_pct
leiden_schpl_qc                                             
3                      7242    10373  0.698159          69.8
7                       686     1748  0.392449          39.2
8                       676     1741  0.388283          38.8
10                      121      363  0.333333          33.3
9                       268      838  0.319809          32.0
6                       551     1857  0.296715          29.7
5                       528     1906  0.277020          27.7
2                      4433    16751  0.264641          26.5
0                      2762    19720  0.140061          14.0
4                      1013     8769  0.

## Section 10: Visualizations

In [37]:
print("=" * 70)
print("STEP 8: VISUALIZATIONS")
print("=" * 70)

print("  Generating 6-panel overview...")
fig, axes = plt.subplots(2, 3, figsize=(24, 16))

# [0,0] scHPL branch assignment
sc.pl.umap(adata_qry, color=COL_SCHPL_BRANCH,
           ax=axes[0, 0], show=False, frameon=False, size=3,
           legend_loc="right margin", legend_fontsize=7,
           title="Branch Used for scHPL")

# [0,1] scANVI final label (from v2.1)
sc.pl.umap(adata_qry, color="cell_type_scarches_final",
           ax=axes[0, 1], show=False, frameon=False, size=3,
           legend_loc="right margin", legend_fontsize=6,
           title="scANVI Final Label (v2.1)")

# [0,2] scHPL cleaned prediction
sc.pl.umap(adata_qry, color=COL_SCHPL_PRED,
           ax=axes[0, 2], show=False, frameon=False, size=3,
           legend_loc="right margin", legend_fontsize=6,
           title="Branch-wise scHPL Prediction")

# [1,0] scHPL rejection type
sc.pl.umap(adata_qry, color=COL_SCHPL_REJ_TYPE,
           ax=axes[1, 0], show=False, frameon=False, size=3,
           legend_loc="right margin", legend_fontsize=7,
           title="Rejection Type (dist / RE / prob / skip)")

# [1,1] scHPL probability
sc.pl.umap(adata_qry, color=COL_SCHPL_PROB,
           ax=axes[1, 1], show=False, frameon=False, size=3,
           cmap="viridis", vmin=0, vmax=1,
           title="scHPL Posterior Probability")

# [1,2] Novel candidate clusters
sc.pl.umap(adata_qry, color="schpl_novel_candidate",
           ax=axes[1, 2], show=False, frameon=False, size=3,
           title="Novel Candidate Cells")

plt.suptitle(f"Stromal treeArches / scHPL {PIPELINE_VERSION}", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / f"schpl_overview_6panel_{PIPELINE_TAG}.{FIG_FORMAT}",
            dpi=DPI, bbox_inches="tight")
plt.close('all')
print(f"  [OK] schpl_overview_6panel_{PIPELINE_TAG}.{FIG_FORMAT}")

branch_fig_dir = FIG_DIR / "branches"
branch_fig_dir.mkdir(parents=True, exist_ok=True)
print("\n  Generating branch-specific overview figures...")
for branch_name, cfg in BRANCH_CONFIGS.items():
    mask_branch = adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == branch_name
    n_branch = int(mask_branch.sum())
    if n_branch == 0:
        print(f"    [INFO] {branch_name}: 0 cells, skip branch figure")
        continue

    ad_branch = adata_qry[mask_branch].copy()
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    axes = axes.ravel()

    sc.pl.umap(ad_branch, color="cell_type_scarches_final",
               ax=axes[0], show=False, frameon=False, size=6,
               legend_loc="right margin", legend_fontsize=7,
               title=f"{cfg['display']}: scANVI Final")
    sc.pl.umap(ad_branch, color=COL_SCHPL_PRED,
               ax=axes[1], show=False, frameon=False, size=6,
               legend_loc="right margin", legend_fontsize=7,
               title=f"{cfg['display']}: scHPL Prediction")
    sc.pl.umap(ad_branch, color=COL_SCHPL_REJ_TYPE,
               ax=axes[2], show=False, frameon=False, size=6,
               legend_loc="right margin", legend_fontsize=7,
               title=f"{cfg['display']}: Rejection Type")
    sc.pl.umap(ad_branch, color=COL_SCHPL_PROB,
               ax=axes[3], show=False, frameon=False, size=6,
               cmap="viridis", vmin=0, vmax=1,
               title=f"{cfg['display']}: scHPL Probability")

    rej_rate_branch = np.nan
    if branch_name in branch_rejection_summary.index and branch_rejection_summary.loc[branch_name, 'n_modeled'] > 0:
        rej_rate_branch = float(branch_rejection_summary.loc[branch_name, 'rej_rate_pct'])

    plt.suptitle(
        f"{cfg['display']} branch | n={n_branch:,} | reject={rej_rate_branch:.1f}%",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    branch_fig_path = branch_fig_dir / f"schpl_branch_overview_{branch_name}_{PIPELINE_TAG}.{FIG_FORMAT}"
    fig.savefig(branch_fig_path, dpi=DPI, bbox_inches="tight")
    plt.close('all')
    print(f"    [OK] {branch_name}: {branch_fig_path.name}")

STEP 8: VISUALIZATIONS
  Generating 6-panel overview...


  [OK] schpl_overview_6panel_v1_1_branchwise.pdf

  Generating branch-specific overview figures...
    [OK] endothelial: schpl_branch_overview_endothelial_v1_1_branchwise.pdf
    [OK] fibroblast: schpl_branch_overview_fibroblast_v1_1_branchwise.pdf
    [OK] smc: schpl_branch_overview_smc_v1_1_branchwise.pdf


In [31]:
print("  Generating scANVI vs scHPL comparison...")
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sc.pl.umap(adata_qry, color="cell_type_scarches_final",
           ax=axes[0], show=False, frameon=False, size=4,
           legend_loc="right margin", legend_fontsize=7,
           title="scANVI Final Label (v2.1, conf>=0.5)")

sc.pl.umap(adata_qry, color=COL_SCHPL_PRED,
           ax=axes[1], show=False, frameon=False, size=4,
           legend_loc="right margin", legend_fontsize=7,
           title="Branch-wise scHPL Prediction")

plt.tight_layout()
fig.savefig(FIG_DIR / f"schpl_vs_scanvi_sidebyside_{PIPELINE_TAG}.{FIG_FORMAT}",
            dpi=DPI, bbox_inches="tight")
plt.close('all')
print(f"  [OK] schpl_vs_scanvi_sidebyside_{PIPELINE_TAG}.{FIG_FORMAT}")

  Generating scANVI vs scHPL comparison...


  [OK] schpl_vs_scanvi_sidebyside_v1_1_branchwise.pdf


In [32]:
print("  Generating rejection confidence analysis...")
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# [0] scHPL probability histogram: accepted vs rejected
if adata_qry.obs[COL_SCHPL_PROB].notna().any():
    prob_acc = adata_qry.obs.loc[
        (~adata_qry.obs[COL_SCHPL_REJECTED]) & (adata_qry.obs[COL_SCHPL_PRED] != "Ignored"),
        COL_SCHPL_PROB,
    ]
    prob_rej = adata_qry.obs.loc[
        adata_qry.obs[COL_SCHPL_REJECTED],
        COL_SCHPL_PROB,
    ]
    axes[0].hist(prob_acc.dropna(), bins=50, alpha=0.7, color="steelblue", label="Accepted")
    axes[0].hist(prob_rej.dropna(), bins=50, alpha=0.7, color="tomato", label="Rejected")
    axes[0].axvline(SCHPL_REJ_THRESHOLD, color="red", linestyle="--",
                    label=f"rej_threshold={SCHPL_REJ_THRESHOLD}")
    axes[0].set_xlabel("scHPL Posterior Probability", fontsize=12)
    axes[0].set_ylabel("Cell Count", fontsize=12)
    axes[0].set_title("Probability: Accepted vs Rejected", fontsize=12)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
else:
    axes[0].text(0.5, 0.5, "No probability output", ha="center", va="center")
    axes[0].set_axis_off()

rej_type_counts = adata_qry.obs[COL_SCHPL_REJ_TYPE].value_counts()
colors_pie = ["steelblue" if t == "accepted" else "tomato" for t in rej_type_counts.index]
axes[1].pie(
    rej_type_counts.values,
    labels=rej_type_counts.index,
    autopct="%1.1f%%",
    colors=colors_pie,
    startangle=90,
)
axes[1].set_title("Rejection / Skip Type Distribution", fontsize=12)

conf_acc = adata_qry.obs.loc[
    (~adata_qry.obs[COL_SCHPL_REJECTED]) & (adata_qry.obs[COL_SCHPL_PRED] != "Ignored"),
    "scarches_confidence",
]
conf_rej = adata_qry.obs.loc[
    adata_qry.obs[COL_SCHPL_REJECTED],
    "scarches_confidence",
]

axes[2].hist(conf_acc, bins=50, alpha=0.7, color="steelblue", label="scHPL Accepted",
             density=True)
axes[2].hist(conf_rej, bins=50, alpha=0.7, color="tomato", label="scHPL Rejected",
             density=True)
axes[2].axvline(0.5, color="black", linestyle="--", label="scANVI threshold=0.5")
axes[2].set_xlabel("scANVI Confidence (v2.1)", fontsize=12)
axes[2].set_ylabel("Density", fontsize=12)
axes[2].set_title("scANVI Confidence for scHPL Accepted/Rejected", fontsize=12)
axes[2].legend()
axes[2].grid(alpha=0.3)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIG_DIR / f"schpl_rejection_analysis_{PIPELINE_TAG}.{FIG_FORMAT}",
            dpi=DPI, bbox_inches="tight")
plt.close('all')
print(f"  [OK] schpl_rejection_analysis_{PIPELINE_TAG}.{FIG_FORMAT}")

  Generating rejection confidence analysis...
  [OK] schpl_rejection_analysis_v1_1_branchwise.pdf


In [33]:
print("  Generating comparison heatmap (accepted modeled cells)...")

df_heat = adata_qry.obs.loc[
    (~adata_qry.obs[COL_SCHPL_REJECTED]) & (adata_qry.obs[COL_SCHPL_PRED] != "Ignored"),
    ["cell_type_scarches_pred", COL_SCHPL_PRED]
].copy()
df_heat.columns = ["scANVI_pred", "scHPL_pred"]

if df_heat.empty:
    print("  [INFO] No accepted modeled cells available; skipping heatmap.")
else:
    ct = pd.crosstab(
        df_heat["scANVI_pred"],
        df_heat["scHPL_pred"],
        normalize="index"
    )

    fig, ax = plt.subplots(figsize=(max(10, ct.shape[1] * 0.8),
                                    max(8, ct.shape[0] * 0.6)))
    sns.heatmap(
        ct,
        ax=ax,
        cmap="Blues",
        annot=ct.shape[0] <= 20,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "Fraction of scANVI cells"},
    )
    ax.set_xlabel("scHPL Prediction", fontsize=12)
    ax.set_ylabel("scANVI Prediction", fontsize=12)
    ax.set_title(
        "scANVI vs branch-wise scHPL: Agreement Heatmap (accepted cells, row-normalized)",
        fontsize=12,
    )
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"schpl_scanvi_comparison_heatmap_{PIPELINE_TAG}.{FIG_FORMAT}",
                dpi=DPI, bbox_inches="tight")
    plt.close('all')
    print(f"  [OK] schpl_scanvi_comparison_heatmap_{PIPELINE_TAG}.{FIG_FORMAT}")

  Generating comparison heatmap (accepted modeled cells)...


  [OK] schpl_scanvi_comparison_heatmap_v1_1_branchwise.pdf


In [34]:
n_novel = adata_qry.obs["schpl_novel_candidate"].sum()
if n_novel > 0:
    print(f"  Generating novel candidate cluster plot ({n_novel:,} cells)...")
    fig, axes = plt.subplots(1, 3, figsize=(21, 7))

    sc.pl.umap(adata_qry, color="schpl_novel_candidate",
               ax=axes[0], show=False, frameon=False, size=4,
               title=f"Novel Candidates (n={n_novel:,})")

    sc.pl.umap(adata_qry, color="leiden_schpl_qc",
               ax=axes[1], show=False, frameon=False, size=4,
               legend_loc="right margin", legend_fontsize=7,
               title="Leiden Clusters (res=0.5)")

    sc.pl.umap(adata_qry, color="scarches_confidence",
               ax=axes[2], show=False, frameon=False, size=4,
               cmap="RdYlGn", vmin=0, vmax=1,
               title="scANVI Confidence")

    plt.tight_layout()
    fig.savefig(FIG_DIR / f"schpl_novel_candidates_{PIPELINE_TAG}.{FIG_FORMAT}",
                dpi=DPI, bbox_inches="tight")
    plt.close('all')
    print(f"  [OK] schpl_novel_candidates_{PIPELINE_TAG}.{FIG_FORMAT}")
else:
    print("  [INFO] No novel candidate clusters detected; skipping Panel 5.")

  Generating novel candidate cluster plot (7,242 cells)...
  [OK] schpl_novel_candidates_v1_1_branchwise.pdf


## Section 11: Save Outputs

In [38]:
print("=" * 70)
print("STEP 9: SAVE OUTPUTS")
print("=" * 70)

label_csv = OUTPUT_DIR / f"schpl_label_comparison_{PIPELINE_TAG}.csv"
df_compare.to_csv(label_csv)
print(f"  [OK] Saved comparison CSV: {label_csv}")

cluster_csv = OUTPUT_DIR / f"cluster_rejection_summary_{PIPELINE_TAG}.csv"
cluster_rej.to_csv(cluster_csv)
print(f"  [OK] Saved cluster rejection summary: {cluster_csv}")

branch_training_csv = OUTPUT_DIR / f"branch_training_summary_{PIPELINE_TAG}.csv"
branch_prediction_csv = OUTPUT_DIR / f"branch_prediction_summary_{PIPELINE_TAG}.csv"
branch_rejection_csv = OUTPUT_DIR / f"branch_rejection_summary_{PIPELINE_TAG}.csv"
branch_agreement_csv = OUTPUT_DIR / f"branch_agreement_summary_{PIPELINE_TAG}.csv"

branch_training_df.to_csv(branch_training_csv, index=False)
branch_prediction_df.to_csv(branch_prediction_csv, index=False)
branch_rejection_summary.to_csv(branch_rejection_csv)
branch_agreement_df.to_csv(branch_agreement_csv, index=False)

print(f"  [OK] Saved branch training summary : {branch_training_csv}")
print(f"  [OK] Saved branch prediction summary: {branch_prediction_csv}")
print(f"  [OK] Saved branch rejection summary : {branch_rejection_csv}")
print(f"  [OK] Saved branch agreement summary : {branch_agreement_csv}")


def _sanitize_df_for_h5ad(df):
    for col in df.columns:
        s = df[col]
        if s.dtype != object:
            continue
        non_na = s.dropna()
        if len(non_na) == 0:
            df[col] = s.fillna("").astype(str)
            continue
        if non_na.map(lambda x: isinstance(x, (bool, np.bool_))).all():
            df[col] = s.fillna(False).astype(np.int8)
            continue
        if non_na.map(lambda x: not isinstance(x, str)).any():
            df[col] = s.map(lambda x: "" if pd.isna(x) else str(x))


branch_h5ad_dir = OUTPUT_DIR / "branch_h5ad"
branch_h5ad_dir.mkdir(parents=True, exist_ok=True)
branch_fig_dir = FIG_DIR / "branches"
branch_exports = {}

print("\n  Saving branch-specific h5ad outputs...")
for branch_name, cfg in BRANCH_CONFIGS.items():
    mask_branch = adata_qry.obs[COL_SCHPL_BRANCH].astype(str) == branch_name
    n_branch = int(mask_branch.sum())
    if n_branch == 0:
        print(f"    [INFO] {branch_name}: 0 cells, skip branch h5ad")
        continue

    ad_branch = adata_qry[mask_branch].copy()
    _sanitize_df_for_h5ad(ad_branch.obs)
    _sanitize_df_for_h5ad(ad_branch.var)
    if ad_branch.raw is not None:
        _sanitize_df_for_h5ad(ad_branch.raw.var)

    branch_h5ad = branch_h5ad_dir / f"adata_stromal_query_schpl_{branch_name}_{PIPELINE_TAG}.h5ad"
    ad_branch.write_h5ad(branch_h5ad, compression="gzip")
    rej_rate_branch = np.nan
    if branch_name in branch_rejection_summary.index and branch_rejection_summary.loc[branch_name, "n_modeled"] > 0:
        rej_rate_branch = float(branch_rejection_summary.loc[branch_name, "rej_rate_pct"])

    branch_exports[branch_name] = {
        "display": cfg["display"],
        "h5ad": str(branch_h5ad),
        "n_cells": int(ad_branch.n_obs),
        "n_rejected": int(ad_branch.obs[COL_SCHPL_REJECTED].sum()),
        "rej_rate_pct": rej_rate_branch,
        "figure": str(branch_fig_dir / f"schpl_branch_overview_{branch_name}_{PIPELINE_TAG}.{FIG_FORMAT}"),
    }
    print(
        f"    [OK] {branch_name}: {branch_h5ad.name} | "
        f"n={ad_branch.n_obs:,} | reject={rej_rate_branch:.1f}%"
    )

adata_qry.uns["schpl_mapping"] = {
    "version": PIPELINE_VERSION,
    "tag": PIPELINE_TAG,
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "method": "treeArches latent + branch-wise scHPL",
    "reference_manifest": str(PATH_REF_MANIFEST) if PATH_REF_MANIFEST.exists() else None,
    "legacy_reference_file": str(PATH_REF_H5AD) if PATH_REF_H5AD.exists() else None,
    "query_file": str(PATH_QRY_H5AD),
    "query_manifest": str(PATH_QRY_MANIFEST) if PATH_QRY_MANIFEST.exists() else None,
    "latent_key": QRY_LATENT_KEY,
    "reference_label_key": REF_LABEL_KEY,
    "n_ref_cells_total": int(ref_total_cells),
    "n_ref_cells_modeled": int(ref_modeled_cells),
    "n_qry_cells": int(adata_qry.n_obs),
    "n_modeled_qry_cells": int((adata_qry.obs[COL_SCHPL_PRED] != "Ignored").sum()),
    "n_ignored_qry_cells": int((adata_qry.obs[COL_SCHPL_PRED] == "Ignored").sum()),
    "agreement_pct_on_accepted": None if pd.isna(agree_rate) else float(agree_rate),
    "n_accepted": int((adata_qry.obs[COL_SCHPL_PRED] != "Ignored").sum() - adata_qry.obs[COL_SCHPL_REJECTED].sum()),
    "n_rejected": int(adata_qry.obs[COL_SCHPL_REJECTED].sum()),
    "schpl_rej_threshold": float(SCHPL_REJ_THRESHOLD),
    "branch_output_dir": str(branch_h5ad_dir),
    "branch_figure_dir": str(branch_fig_dir),
    "branch_exports": branch_exports,
    "branches": {
        branch_name: {
            "display": cfg["display"],
            "query_subset": cfg["query_subset"],
            "allowed_l2": list(cfg["allowed_l2"]),
            "reference_h5ad": str(ref_branch_data[branch_name]["h5ad_path"]) if branch_name in ref_branch_data else None,
            "tree_path": str(trained_tree_paths.get(branch_name, "")),
        }
        for branch_name, cfg in BRANCH_CONFIGS.items()
    },
}

out_h5ad = OUTPUT_DIR / f"adata_stromal_query_schpl_{PIPELINE_TAG}.h5ad"
adata_qry.write_h5ad(out_h5ad, compression="gzip")
file_size = out_h5ad.stat().st_size / 1024**2
print(f"  [OK] Saved query h5ad: {out_h5ad} ({file_size:.1f} MB)")

cfg_json = OUTPUT_DIR / f"schpl_config_{PIPELINE_TAG}.json"
with open(cfg_json, "w") as f:
    json.dump({
        "version": PIPELINE_VERSION,
        "reference_manifest": str(PATH_REF_MANIFEST),
        "legacy_reference_h5ad": str(PATH_REF_H5AD),
        "query_h5ad": str(PATH_QRY_H5AD),
        "query_manifest": str(PATH_QRY_MANIFEST),
        "output_h5ad": str(out_h5ad),
        "branch_output_dir": str(branch_h5ad_dir),
        "branch_figure_dir": str(branch_fig_dir),
        "branch_exports": branch_exports,
        "tree_paths": {k: str(v) for k, v in trained_tree_paths.items()},
        "reference_h5ads": {k: str(v["h5ad_path"]) for k, v in ref_branch_data.items()},
        "comparison_csv": str(label_csv),
        "cluster_csv": str(cluster_csv),
        "branch_training_csv": str(branch_training_csv),
        "branch_prediction_csv": str(branch_prediction_csv),
        "branch_rejection_csv": str(branch_rejection_csv),
        "branch_agreement_csv": str(branch_agreement_csv),
        "fig_dir": str(FIG_DIR),
    }, f, indent=2)
print(f"  [OK] Saved config JSON: {cfg_json}")

STEP 9: SAVE OUTPUTS


  [OK] Saved comparison CSV: /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/schpl_label_comparison_v1_1_branchwise.csv
  [OK] Saved cluster rejection summary: /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/cluster_rejection_summary_v1_1_branchwise.csv
  [OK] Saved branch training summary : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/branch_training_summary_v1_1_branchwise.csv
  [OK] Saved branch prediction summary: /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/branch_prediction_summary_v1_1_branchwise.csv
  [OK] Saved branch rejection summary : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/branch_rejection_summary_v1_1_branchwise.csv
  [OK] Saved branch agreement summary : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/branch_agreement_summary_v1_1_branchwise.csv

  Saving branch-specific h5ad outputs...
    [OK] endothelial: adata_stromal_query_schpl_endothelial_v1_1_branchwise.h5ad | n=29,960 | reject=42.6%
    [OK] fibroblast: ada

## Section 12: Final Summary

In [39]:
print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

print(f"\nVersion          : {PIPELINE_VERSION}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Figures directory: {FIG_DIR}")
print(f"Tree directory   : {TREE_DIR}")
print(f"Branch h5ad dir  : {OUTPUT_DIR / 'branch_h5ad'}")
print(f"Branch fig dir   : {FIG_DIR / 'branches'}")

print(f"\nMain output h5ad : {OUTPUT_DIR / f'adata_stromal_query_schpl_{PIPELINE_TAG}.h5ad'}")
print(f"Config JSON      : {OUTPUT_DIR / f'schpl_config_{PIPELINE_TAG}.json'}")
print(f"Comparison CSV   : {OUTPUT_DIR / f'schpl_label_comparison_{PIPELINE_TAG}.csv'}")
print(f"Cluster CSV      : {OUTPUT_DIR / f'cluster_rejection_summary_{PIPELINE_TAG}.csv'}")
print(f"Branch reject CSV: {OUTPUT_DIR / f'branch_rejection_summary_{PIPELINE_TAG}.csv'}")

print("\nBranch tree files:")
for branch_name, tree_path in trained_tree_paths.items():
    print(f"  {branch_name:<12} : {tree_path}")

print("\nBranch-specific outputs:")
for branch_name in BRANCH_CONFIGS:
    info = adata_qry.uns.get("schpl_mapping", {}).get("branch_exports", {}).get(branch_name)
    if not info:
        print(f"  {branch_name:<12} : missing")
        continue
    print(f"  {branch_name:<12} : h5ad={info['h5ad']}")
    print(f"  {'':14} figure={info['figure']}")

print("\nKey statistics:")
print(f"  Query cells                : {adata_qry.n_obs:,}")
print(f"  Query HVGs                 : {adata_qry.n_vars:,}")
print(f"  Modeled by scHPL           : {(adata_qry.obs[COL_SCHPL_PRED] != 'Ignored').sum():,}")
print(f"  Ignored by scHPL           : {(adata_qry.obs[COL_SCHPL_PRED] == 'Ignored').sum():,}")
print(f"  Accepted by scHPL          : {((adata_qry.obs[COL_SCHPL_PRED] != 'Ignored') & (~adata_qry.obs[COL_SCHPL_REJECTED])).sum():,}")
print(f"  Rejected by scHPL          : {adata_qry.obs[COL_SCHPL_REJECTED].sum():,}")
print(f"  Agreement with scANVI      : {('NA' if pd.isna(agree_rate) else f'{agree_rate:.1f}%')} (accepted only)")
print(f"  Novel candidate clusters   : {len(novel_clusters)}")
print(f"  Novel candidate cells      : {adata_qry.obs['schpl_novel_candidate'].sum():,}")

print("\nAccepted-cell agreement by branch:")
print(branch_agreement_df.to_string(index=False))

print("\nDone.")


PIPELINE COMPLETE

Version          : v1.1-branchwise
Output directory : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise
Figures directory: /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/figures
Tree directory   : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/trees
Branch h5ad dir  : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/branch_h5ad
Branch fig dir   : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/figures/branches

Main output h5ad : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/adata_stromal_query_schpl_v1_1_branchwise.h5ad
Config JSON      : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/schpl_config_v1_1_branchwise.json
Comparison CSV   : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/schpl_label_comparison_v1_1_branchwise.csv
Cluster CSV      : /home/h2048/data/py/0406/stromal_schpl_v1_1_branchwise/cluster_rejection_summary_v1_1_branchwise.csv
Branch reject CSV: /home/h2048/data/py/0406/stromal_schpl_v1_1_br